# **Dicoding Indonesia: Belajar Fundamental Deep Learning**
# Proyek NLP Analisis Sentiment 
**Analisis sentimen review aplikasi game Mobile Legends pada Play Store dengan Pendekatan Random Forrest dan Super Vector Machince (SVM).**

# 0.Import Library

In [ ]:
import re  
import csv
import nltk 
import requests
import numpy as np
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt

from io import StringIO
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize  
from sklearn.metrics import accuracy_score
from Sastrawi.Stemmer.StemmerFactory import (StemmerFactory) 
from sklearn.feature_extraction.text import TfidfVectorizer

# python -m pip install sastrawi
nltk.download("punkt_tab")  
nltk.download("stopwords") 

[nltk_data] Downloading package punkt_tab to C:\Users\Febryo Fibonacci
[nltk_data]     A\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Febryo Fibonacci
[nltk_data]     A\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# 2.Loading Dataset

In [ ]:
df = pd.DataFrame("Ulasan_APK_Mobile_Legends.csv")
df.head(-10)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,a5f8f395-4d19-413c-84a3-6c5bb9065d21,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"hallo admin yth, tolong perbaiki masalah solo ...",3,2,2.1.95.12053,2026-08-16 15:54:09,NaN,NaT,2.1.95.12053
1,02c9b2d8-fa0e-4644-b506-10fddd7aed4a,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Sebenarnya udah bagus banget, dari Matchmaking...",5,7,2.1.95.12053,2026-08-13 19:05:53,NaN,NaT,2.1.95.12053
2,c66b5494-4ed1-45af-85b3-6dc752e590d1,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Kpd moonton terhormat. Banyak sekali komplen s...,1,3,2.1.95.12053,2026-08-20 16:59:53,NaN,NaT,2.1.95.12053
3,76c2e484-e9a3-40ad-b353-079132d5b031,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"bukan maksud untuk menjelekan, tapi sering sek...",3,3,2.1.95.12053,2026-08-20 20:58:02,NaN,NaT,2.1.95.12053
4,75ff1320-d876-499c-9543-fe80999f9783,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"game jelek, gak recomended, sinyal Wifi diruma...",1,3,2.1.95.12053,2026-08-17 14:40:15,"Halo Kak,\nMaaf atas masalah jaringan yang Kak...",2026-08-17 16:17:22,2.1.95.12053
...,...,...,...,...,...,...,...,...,...,...,...
341985,9e837ce3-840e-4aa4-bf31-7fb8e15e6353,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Semakin lama semakin kocak ini apk, jaringan 4...",1,0,1.5.38.5881,2023-06-10 14:35:28,NaN,NaT,1.5.38.5881
341986,d964ecac-6bbb-4ee6-b719-87da27573a9d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Game tidak bermutu Lebih banyak kalah dan tim ...,1,0,1.9.43.10342,2025-01-12 03:04:40,NaN,NaT,1.9.43.10342
341987,af32fbac-46f2-488e-91ec-049731965d15,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Makin kesini makin naik aja udah gak kuat hap ...,1,0,NaN,2023-09-01 04:29:48,NaN,NaT,NaN
341988,36aac986-c81f-487d-a2f7-babd0b7a7309,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Bukan gamenya yg rusak tetapi HP kalian semua ...,5,0,1.7.26.7851,2022-10-29 20:39:35,NaN,NaT,1.7.26.7851


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 342000 entries, 0 to 341999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              342000 non-null  str           
 1   userName              342000 non-null  str           
 2   userImage             342000 non-null  str           
 3   content               342000 non-null  str           
 4   score                 342000 non-null  int64         
 5   thumbsUpCount         342000 non-null  int64         
 6   reviewCreatedVersion  254480 non-null  str           
 7   at                    342000 non-null  datetime64[us]
 8   replyContent          21838 non-null   str           
 9   repliedAt             21838 non-null   datetime64[us]
 10  appVersion            254480 non-null  str           
dtypes: datetime64[us](2), int64(2), str(7)
memory usage: 28.7 MB


In [5]:
df.shape

(342000, 11)

In [6]:
def get_missing_values(data):
    missing = pd.DataFrame({
        "Missing Values": data.isna().sum(),
        "Perncentage": (data.isna().sum() / len(data) * 100).round(2) 
    })

    return missing

def get_duplicated_row(data):
    print("Duplicated Rows:", data.duplicated)

In [7]:
get_missing_values(df)

,Missing Values,Perncentage
reviewId,0,0.00
userName,0,0.00
userImage,0,0.00
content,0,0.00
score,0,0.00
thumbsUpCount,0,0.00
reviewCreatedVersion,87520,25.59
at,0,0.00
replyContent,320162,93.61
repliedAt,320162,93.61


In [8]:
get_duplicated_row(df)

Duplicated Rows: <bound method DataFrame.duplicated of                                     reviewId         userName  \
0       a5f8f395-4d19-413c-84a3-6c5bb9065d21  Pengguna Google   
1       02c9b2d8-fa0e-4644-b506-10fddd7aed4a  Pengguna Google   
2       c66b5494-4ed1-45af-85b3-6dc752e590d1  Pengguna Google   
3       76c2e484-e9a3-40ad-b353-079132d5b031  Pengguna Google   
4       75ff1320-d876-499c-9543-fe80999f9783  Pengguna Google   
...                                      ...              ...   
341995  68cd8887-1c5c-462d-980d-f454cb1d2223  Pengguna Google   
341996  2a75ea48-8ecc-4688-a437-8c6c2f3d05d4  Pengguna Google   
341997  a607b92d-25ca-4c25-837b-2a44c7fb49bd  Pengguna Google   
341998  d0135776-4255-45c6-b9a6-8b4bee97337f  Pengguna Google   
341999  0938e59a-5e54-409f-b4f9-772f6a6b8ee0  Pengguna Google   

                                                userImage  \
0       https://play-lh.googleusercontent.com/EGemoI2N...   
1       https://play-lh.googleusercontent.

In [9]:
# Clean missing values dan duplicated rows
df_clean = df.dropna()
df_clean = df_clean.drop_duplicates()

In [10]:
get_missing_values(df_clean)

,Missing Values,Perncentage
reviewId,0,0.0
userName,0,0.0
userImage,0,0.0
content,0,0.0
score,0,0.0
thumbsUpCount,0,0.0
reviewCreatedVersion,0,0.0
at,0,0.0
replyContent,0,0.0
repliedAt,0,0.0


In [11]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 17589 entries, 4 to 341856
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              17589 non-null  str           
 1   userName              17589 non-null  str           
 2   userImage             17589 non-null  str           
 3   content               17589 non-null  str           
 4   score                 17589 non-null  int64         
 5   thumbsUpCount         17589 non-null  int64         
 6   reviewCreatedVersion  17589 non-null  str           
 7   at                    17589 non-null  datetime64[us]
 8   replyContent          17589 non-null  str           
 9   repliedAt             17589 non-null  datetime64[us]
 10  appVersion            17589 non-null  str           
dtypes: datetime64[us](2), int64(2), str(7)
memory usage: 1.6 MB


# 3.Text Preprocessing

In [12]:
# Case folding text
df_clean["text_lowercase"] = df["content"].str.lower()
df_clean["text_lowercase"]

4         game jelek, gak recomended, sinyal wifi diruma...
15        sistem apaan ini?? setim dengan player rank le...
16        saya mengajukan keberatan atas dugaan pengguna...
17        tolong fitur highlight nya dibikin otomatis te...
33        tolong banget ya, kenapa ini banyak komputer a...
                                ...                        
341536    untuk jaringan tolong d perbaiki masa sinyal t...
341778    mats tidak seimbang lawan main nya di atas rat...
341836    pembagian tim ngawur, tim sendiri gabisa maen,...
341854    moonton tolong la perbaiki sistem pertandingan...
341856    akun dulu tidak bisa kembali, padahal sudah sa...
Name: text_lowercase, Length: 17589, dtype: str

In [13]:
df_clean["text_clean"] = df_clean.apply(lambda x: re.sub(r"^[^-]*-\s", "", x["text_lowercase"]), axis=1)
df_clean["text_clean"]

4         game jelek, gak recomended, sinyal wifi diruma...
15        sistem apaan ini?? setim dengan player rank le...
16        saya mengajukan keberatan atas dugaan pengguna...
17        tolong fitur highlight nya dibikin otomatis te...
33        tolong banget ya, kenapa ini banyak komputer a...
                                ...                        
341536    untuk jaringan tolong d perbaiki masa sinyal t...
341778    mats tidak seimbang lawan main nya di atas rat...
341836    pembagian tim ngawur, tim sendiri gabisa maen,...
341854    moonton tolong la perbaiki sistem pertandingan...
341856    akun dulu tidak bisa kembali, padahal sudah sa...
Name: text_clean, Length: 17589, dtype: str

In [14]:
# Delete mention
df_clean["text_clean"] = df_clean.apply(lambda x: re.sub(r"@[A-Za-z0-9]+", "", x["text_clean"]), axis=1)

# Delete hashtag
df_clean["text_clean"] = df_clean.apply(lambda x: re.sub(r"#[A-Za-z0-9]+", "", x["text_clean"]), axis=1)

# Delete RT
df_clean["text_clean"] = df_clean.apply(lambda x: re.sub(r"RT[\s]", "", x["text_clean"]), axis=1)

# Delete link
df_clean["text_clean"] = df_clean.apply(lambda x: re.sub(r"http\S+", "", x["text_clean"]), axis=1)

# Delete angka
df_clean["text_clean"] = df_clean.apply(lambda x: re.sub(r"[0-9]+", "", x["text_clean"]), axis=1)

# Delete punctuation
df_clean["text_clean"] = df_clean.apply(lambda x: re.sub(r"[^\w\s]", "", x["text_clean"]), axis=1)


df_clean["text_clean"]

4         game jelek gak recomended sinyal wifi dirumah ...
15        sistem apaan ini setim dengan player rank lebi...
16        saya mengajukan keberatan atas dugaan pengguna...
17        tolong fitur highlight nya dibikin otomatis te...
33        tolong banget ya kenapa ini banyak komputer ai...
                                ...                        
341536    untuk jaringan tolong d perbaiki masa sinyal t...
341778    mats tidak seimbang lawan main nya di atas rat...
341836    pembagian tim ngawur tim sendiri gabisa maenda...
341854    moonton tolong la perbaiki sistem pertandingan...
341856    akun dulu tidak bisa kembali padahal sudah say...
Name: text_clean, Length: 17589, dtype: str

In [15]:
# slangwords
# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
slangwords = {
    "@": "di",
    "abis": "habis",
    "wtb": "beli",
    "masi": "masih",
    "wts": "jual",
    "wtt": "tukar",
    "bgt": "banget",
    "maks": "maksimal",
}

df_clean["text_clean"] = df_clean.apply(lambda x: " ".join([slangwords.get(word.lower(), word) for word in x["text_clean"].split()]),axis=1)
df_clean["text_clean"]

4         game jelek gak recomended sinyal wifi dirumah ...
15        sistem apaan ini setim dengan player rank lebi...
16        saya mengajukan keberatan atas dugaan pengguna...
17        tolong fitur highlight nya dibikin otomatis te...
33        tolong banget ya kenapa ini banyak komputer ai...
                                ...                        
341536    untuk jaringan tolong d perbaiki masa sinyal t...
341778    mats tidak seimbang lawan main nya di atas rat...
341836    pembagian tim ngawur tim sendiri gabisa maenda...
341854    moonton tolong la perbaiki sistem pertandingan...
341856    akun dulu tidak bisa kembali padahal sudah say...
Name: text_clean, Length: 17589, dtype: str

In [16]:
# Remove stopwords
stopwords_id = stopwords.words("indonesian")
stopword_tambahan = [
            "iya",
            "yaa",
            "gak",
            "nya",
            "na",
            "sih",
            "ku",
            "di",
            "ga",
            "ya",
            "gaa",
            "loh",
            "kah",
            "woi",
            "woii",
            "woy",
        ]
stopwords_id.extend(stopword_tambahan)
stopwords_id

['ada',
 'adalah',
 'adanya',
 'adapun',
 'agak',
 'agaknya',
 'agar',
 'akan',
 'akankah',
 'akhir',
 'akhiri',
 'akhirnya',
 'aku',
 'akulah',
 'amat',
 'amatlah',
 'anda',
 'andalah',
 'antar',
 'antara',
 'antaranya',
 'apa',
 'apaan',
 'apabila',
 'apakah',
 'apalagi',
 'apatah',
 'artinya',
 'asal',
 'asalkan',
 'atas',
 'atau',
 'ataukah',
 'ataupun',
 'awal',
 'awalnya',
 'bagai',
 'bagaikan',
 'bagaimana',
 'bagaimanakah',
 'bagaimanapun',
 'bagi',
 'bagian',
 'bahkan',
 'bahwa',
 'bahwasanya',
 'baik',
 'bakal',
 'bakalan',
 'balik',
 'banyak',
 'bapak',
 'baru',
 'bawah',
 'beberapa',
 'begini',
 'beginian',
 'beginikah',
 'beginilah',
 'begitu',
 'begitukah',
 'begitulah',
 'begitupun',
 'bekerja',
 'belakang',
 'belakangan',
 'belum',
 'belumlah',
 'benar',
 'benarkah',
 'benarlah',
 'berada',
 'berakhir',
 'berakhirlah',
 'berakhirnya',
 'berapa',
 'berapakah',
 'berapalah',
 'berapapun',
 'berarti',
 'berawal',
 'berbagai',
 'berdatangan',
 'beri',
 'berikan',
 'berikut'

In [17]:
df_clean["text_clean"] = df_clean.apply(lambda x: " ".join([word for word in x["text_clean"].split() if word not in (stopwords_id)]), axis=1)
df_clean["text_clean"]

4         game jelek recomended sinyal wifi dirumah bagu...
15        sistem setim player rank gw main cover tim fig...
16        mengajukan keberatan dugaan penggunaan ai trai...
17        tolong fitur highlight dibikin otomatis tersim...
33        tolong banget komputer ai rank honor main solo...
                                ...                        
341536    jaringan tolong d perbaiki sinyal tibal merah ...
341778    mats seimbang lawan main rataa tim tolong perb...
341836    pembagian tim ngawur tim gabisa maendapet lawa...
341854    moonton tolong la perbaiki sistem pertandingan...
341856                        akun kaitkan akun google play
Name: text_clean, Length: 17589, dtype: str

In [18]:
# Tokenize
df_clean["token"] = df_clean.apply(lambda x: word_tokenize(x["text_clean"]), axis=1)
df_clean["token"]

4         [game, jelek, recomended, sinyal, wifi, diruma...
15        [sistem, setim, player, rank, gw, main, cover,...
16        [mengajukan, keberatan, dugaan, penggunaan, ai...
17        [tolong, fitur, highlight, dibikin, otomatis, ...
33        [tolong, banget, komputer, ai, rank, honor, ma...
                                ...                        
341536    [jaringan, tolong, d, perbaiki, sinyal, tibal,...
341778    [mats, seimbang, lawan, main, rataa, tim, tolo...
341836    [pembagian, tim, ngawur, tim, gabisa, maendape...
341854    [moonton, tolong, la, perbaiki, sistem, pertan...
341856                  [akun, kaitkan, akun, google, play]
Name: token, Length: 17589, dtype: object

In [19]:
# Stemming
factory = StemmerFactory()
stemmer = factory.create_stemmer()

df_clean["stemmed"] = df_clean["token"].apply(lambda tokens: [stemmer.stem(token) for token in tokens])
df_clean["stemmed"]

4         [game, jelek, recomended, sinyal, wifi, rumah,...
15        [sistem, tim, player, rank, gw, main, cover, t...
16        [aju, berat, duga, guna, ai, trainingbot, matc...
17        [tolong, fitur, highlight, bikin, otomatis, si...
33        [tolong, banget, komputer, ai, rank, honor, ma...
                                ...                        
341536    [jaring, tolong, d, baik, sinyal, tibal, merah...
341778    [mats, imbang, lawan, main, rataa, tim, tolong...
341836    [bagi, tim, ngawur, tim, gabisa, maendapet, la...
341854    [moonton, tolong, la, baik, sistem, tanding, k...
341856                     [akun, kait, akun, google, play]
Name: stemmed, Length: 17589, dtype: object

# 4.Labeling

In [20]:
# Membaca data kamus kata-kata positif dari GitHub
lexicon_positive = dict()

# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub
response = requests.get(
    "https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv"
)

response

<Response [200]>

In [21]:
if response.status_code == 200:

    # Membaca teks response dalam bentuk csv dengan pembaca CSV dan pemisah koma
    reader = csv.reader(StringIO(response.text), delimiter=",")

    lexicon_positive = {row[0]: int(row[1]) for row in reader}

else:
    print("Failed to fetch positive lexicon data")

lexicon_positive

{'hai': 3,
 'merekam': 2,
 'ekstensif': 3,
 'paripurna': 1,
 'detail': 2,
 'pernik': 3,
 'belas': 2,
 'welas': 4,
 'kabung': 1,
 'rahayu': 4,
 'maaf': 2,
 'hello': 2,
 'promo': 3,
 'terimakasih': 5,
 'cover': 3,
 'mohon': 2,
 'mengawal': 2,
 'statistik': 1,
 'keluangan': 3,
 'jalan terbuka': 3,
 'banyaknya': 3,
 'lebar': 3,
 'bentang': 1,
 'hendaknya': 1,
 'silahkan': 3,
 'semboyan': 2,
 'ditunggu': 2,
 'akses': 2,
 'penerangan': 2,
 'hi': 1,
 'dibantu': 2,
 'makasih': 4,
 'halo': 1,
 'thanks': 3,
 'pengembangan': 3,
 'diva': 2,
 'punya': 3,
 'tidak segan': 2,
 'detailnya': 1,
 'tak segan': 2,
 'aktivasi': 2,
 'asih': 3,
 'kasih sayang': 5,
 'kekaguman': 4,
 'kehangatan': 4,
 'afeksi': 2,
 'renjana': 2,
 'amor': 2,
 'cinta kasih': 5,
 'tresna': 2,
 'filantropi': 2,
 'cintrong': 2,
 'suasana (hati)': 1,
 'dinamika': 3,
 'tuhan': 3,
 'merespon': 3,
 'makmur': 4,
 'suka cita': 4,
 'pengguna': 1,
 'tunggu': 1,
 'lotre': 2,
 'nggak': 1,
 'kupon': 3,
 'terpelihara': 4,
 'terawat': 5,
 'tersa

In [22]:
# Membaca data kamus kata-kata negatif dari GitHub
lexicon_negative = dict()

response = requests.get(
    "https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv"
)
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub
response

<Response [200]>

In [23]:
if (response.status_code == 200):
    reader = csv.reader(StringIO(response.text), delimiter=",")
    lexicon_negative = {row[0]: int(row[1]) for row in reader}
else:
    print("Failed to fetch negative lexicon data")

lexicon_negative

{'putus tali gantung': -2,
 'gelebah': -2,
 'gobar hati': -2,
 'tersentuh (perasaan)': -1,
 'isak': -5,
 'larat hati': -3,
 'nelangsa': -3,
 'remuk redam': -5,
 'tidak segan': -2,
 'gemar': -1,
 'tak segan': -1,
 'sesal': -4,
 'pengen': -2,
 'penghayatan': -2,
 'absorpsi': -1,
 'linu': -4,
 'salah benang': -1,
 'sakit': -5,
 'lara': -5,
 'zuhud': -1,
 'mencederai': -4,
 'mengingkari': -4,
 'maaf': -3,
 'mengkhianat': -4,
 'mencelakai': -5,
 'mulu': -1,
 'ngga': -2,
 'borong': -1,
 'lever': -2,
 'kasian': -3,
 'gamau': -4,
 'doang': -1,
 'pulas': -1,
 'abis': -2,
 'coba': -1,
 'kangen': -3,
 'kalau': -1,
 'maunya': -1,
 'seandainya': -1,
 'marilah': -1,
 'bener': -1,
 'yaudah': -4,
 'nggak': -3,
 'gatau': -1,
 'apaan': -4,
 'ngakak': -2,
 'atuh': -1,
 'sekali': -1,
 'menarik hati': -1,
 'cedayam': -2,
 'kece': -3,
 'termakan': -1,
 'belom': -1,
 'malem': -1,
 'mencekau': -2,
 'menduga': -1,
 'menyuarakan': -1,
 'memprediksi': -1,
 'membunyikan': -1,
 'menerka': -1,
 'menaksir': -1,
 'me

In [24]:
def sentiment_analysis_lexicon_indonesia(text):
    score = 0

    for word in text:
        if word in lexicon_positive:
            score = score + lexicon_positive[word]

    for word in text:
        if word in lexicon_negative:
            score = score + lexicon_negative[word]

    polarity = ""

    if score > 0:
        polarity = "postive"
    elif score < 0:
        polarity = "negative"
    else:
        polarity = "neutral"
    return score, polarity

In [25]:
df_clean["Label"] = df_clean.apply(lambda x: sentiment_analysis_lexicon_indonesia(x["stemmed"]), axis=1)

In [26]:
df_clean.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,text_lowercase,text_clean,token,stemmed,Label
4,75ff1320-d876-499c-9543-fe80999f9783,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"game jelek, gak recomended, sinyal Wifi diruma...",1,3,2.1.95.12053,2026-08-17 14:40:15,"Halo Kak,\nMaaf atas masalah jaringan yang Kak...",2026-08-17 16:17:22,2.1.95.12053,"game jelek, gak recomended, sinyal wifi diruma...",game jelek recomended sinyal wifi dirumah bagu...,"[game, jelek, recomended, sinyal, wifi, diruma...","[game, jelek, recomended, sinyal, wifi, rumah,...","(-16, negative)"
15,13acfa6f-791b-43c2-80c4-a8bf6876d4af,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Sistem apaan ini?? setim dengan player rank le...,1,28,2.1.95.12053,2026-08-16 03:00:37,"Halo Kak,\nKami berkomitmen untuk menciptakan ...",2026-06-04 10:49:25,2.1.95.12053,sistem apaan ini?? setim dengan player rank le...,sistem setim player rank gw main cover tim fig...,"[sistem, setim, player, rank, gw, main, cover,...","[sistem, tim, player, rank, gw, main, cover, t...","(-10, negative)"
16,04790e53-887b-41c5-9c42-529c0bb79191,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Saya mengajukan keberatan atas dugaan pengguna...,1,4,2.1.95.12053,2026-08-19 08:16:08,"Halo Kak,\nKami berkomitmen untuk menciptakan ...",2026-08-19 10:49:54,2.1.95.12053,saya mengajukan keberatan atas dugaan pengguna...,mengajukan keberatan dugaan penggunaan ai trai...,"[mengajukan, keberatan, dugaan, penggunaan, ai...","[aju, berat, duga, guna, ai, trainingbot, matc...","(-2, negative)"
17,195e4bc2-e0dc-4621-9ad3-fb7dfa362d50,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,tolong fitur highlight nya dibikin otomatis te...,2,3,2.1.95.12053,2026-08-20 10:41:19,"Halo Kak,\nKami mohon maaf atas pengalaman kur...",2026-08-20 14:58:57,2.1.95.12053,tolong fitur highlight nya dibikin otomatis te...,tolong fitur highlight dibikin otomatis tersim...,"[tolong, fitur, highlight, dibikin, otomatis, ...","[tolong, fitur, highlight, bikin, otomatis, si...","(7, postive)"
33,a6f5fb8c-3d10-4dbf-b185-cbffe249546d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"tolong banget ya, kenapa ini banyak komputer a...",1,2,2.1.95.12053,2026-08-21 02:17:47,"Halo Kak,\nKami mohon maaf atas pengalaman kur...",2026-08-21 11:28:44,2.1.95.12053,"tolong banget ya, kenapa ini banyak komputer a...",tolong banget komputer ai rank honor main solo...,"[tolong, banget, komputer, ai, rank, honor, ma...","[tolong, banget, komputer, ai, rank, honor, ma...","(-4, negative)"


# 5.Feature Extraction
**Term Frequency-Inverse Document Frequency**

In [27]:
# TF-IDF
tfidfvec = TfidfVectorizer()
df_clean["stemmed"].iloc[0]
X = df_clean["stemmed"].apply(" ".join)
y = df_clean["Label"].apply(lambda x: x[1])

# X data
X_tfidf = tfidfvec.fit_transform(X)
features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidfvec.get_feature_names_out())
features_df

,aaaaaaaaaaaahhhhhh,aaaaaaampas,aaaaaahhh,aaaaahh,aaah,aaahhh,aaamiiin,aaamiiinn,aaja,aakn,...,zilonk,zipaul,zize,zodiak,zona,zong,zoning,zonk,zuxin,zzz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17584,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17585,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17586,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17587,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


<!-- **Word2Vector** -->

In [28]:
tokenize_data = df_clean["stemmed"];
model = Word2Vec(sentences=tokenize_data, vector_size=100, window=5, min_count=1, workers=4)

X_w2v = np.array([np.mean([model.wv[word] for word in words if word in model.wv],axis=0) for words in tokenize_data])

print(X_w2v.shape)

(17589, 100)


# 6.Modeling

In [29]:
schema_1 = int(len(df_clean) * 0.8) # skema 1: (80% : 20%)
schema_2 = int(len(df_clean) * 0.7) # skema 2: (70% : 30%)

# Splitting data X_tfidf skema 1: (80% : 20%)
X_tfidf_train_1 = X_tfidf[:schema_1]
X_tfidf_test_1 = X_tfidf[schema_1:]
y_train_1 = y.iloc[:schema_1]
y_test_1 = y.iloc[schema_1:]

# Splitting data X_tfidf skema 1: (70% : 30%)
X_tfidf_train_2 = X_tfidf[:schema_2]
X_tfidf_test_2 = X_tfidf[schema_2:]
y_train_2 = y[:schema_2]
y_test_2 = y[schema_2:]

In [30]:
# Splitting data X_w2v skema 1: (80% : 20%)
X_w2v_train_1 = X_w2v[:schema_1]
X_w2v_test_1 = X_w2v[schema_1:]

# Splitting data X_w2v skema 2: (70% : 30%)
X_w2v_train_2 = X_w2v[:schema_2]
X_w2v_test_2 = X_w2v[schema_2:]

**SVM dengan TF-IDF (80/20)**

In [31]:
from sklearn.linear_model import SGDClassifier

svm_1 = SGDClassifier(loss="hinge", random_state=42).fit(X_tfidf_train_1, y_train_1)

y_pred_train_svm_1 = svm_1.predict(X_tfidf_train_1)
y_pred_test_svm_1 = svm_1.predict(X_tfidf_test_1)

accuracy_train_svm_1 = accuracy_score(y_pred_train_svm_1, y_train_1)
accuracy_test_svm_1 = accuracy_score(y_pred_test_svm_1, y_test_1)

print("SVM - accuracy_train_schema_1:", accuracy_train_svm_1)
print("SVM - accuracy_test_schema_1:", accuracy_test_svm_1)

SVM - accuracy_train_schema_1: 0.9360386610759719
SVM - accuracy_test_schema_1: 0.8712336554860717


In [32]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_1, y_pred_test_svm_1, target_names=["negative", "neutral", "positive"]
    )
)

              precision    recall  f1-score   support

    negative       0.87      0.98      0.92      2169
     neutral       0.00      0.00      0.00       249
    positive       0.88      0.86      0.87      1100

    accuracy                           0.87      3518
   macro avg       0.58      0.61      0.60      3518
weighted avg       0.81      0.87      0.84      3518



d:\Machine Learning\Belajar_Fundamental_Deep_Learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Machine Learning\Belajar_Fundamental_Deep_Learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Machine Learning\Belajar_Fundamental_Deep_Learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


**SVM dengan TF-IDF (70/30)**

In [33]:
from sklearn.linear_model import SGDClassifier

svm_2 = SGDClassifier(loss="hinge", random_state=42).fit(X_tfidf_train_2, y_train_2)

y_pred_train_svm_2 = svm_2.predict(X_tfidf_train_2)
y_pred_test_svm_2 = svm_2.predict(X_tfidf_test_2)

accuracy_train_svm_2 = accuracy_score(y_pred_train_svm_2, y_train_2)
accuracy_test_svm_2 = accuracy_score(y_pred_test_svm_2, y_test_2)

print("SVM - accuracy_train_schema_2:", accuracy_train_svm_2)
print("SVM - accuracy_test_schema_2:", accuracy_test_svm_2)

SVM - accuracy_train_schema_2: 0.9428200129954516
SVM - accuracy_test_schema_2: 0.8645063483039606


In [34]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_2, y_pred_test_svm_2, target_names=["negative", "neutral", "positive"]
    )
)

              precision    recall  f1-score   support

    negative       0.86      0.97      0.91      3315
     neutral       0.00      0.00      0.00       388
    positive       0.86      0.86      0.86      1574

    accuracy                           0.86      5277
   macro avg       0.58      0.61      0.59      5277
weighted avg       0.80      0.86      0.83      5277



d:\Machine Learning\Belajar_Fundamental_Deep_Learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Machine Learning\Belajar_Fundamental_Deep_Learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Machine Learning\Belajar_Fundamental_Deep_Learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


**SVM dengan Word2Vector (80/20)**

In [35]:
from sklearn.linear_model import SGDClassifier

svm_3 = SGDClassifier(loss="hinge", random_state=42).fit(X_w2v_train_1, y_train_1)

y_pred_train_svm_3 = svm_3.predict(X_w2v_train_1)
y_pred_test_svm_3 = svm_3.predict(X_w2v_test_1)

accuracy_train_svm_3 = accuracy_score(y_pred_train_svm_3, y_train_1)
accuracy_test_svm_3 = accuracy_score(y_pred_test_svm_3, y_test_1)

print("SVM - accuracy_train_schema_1:", accuracy_train_svm_3)
print("SVM - accuracy_test_schema_1:", accuracy_test_svm_3)

SVM - accuracy_train_schema_1: 0.7668964536990974
SVM - accuracy_test_schema_1: 0.7316657191586129


In [36]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_1, y_pred_test_svm_3, target_names=["negative", "neutral", "positive"]
    )
)

              precision    recall  f1-score   support

    negative       0.76      0.88      0.82      2169
     neutral       1.00      0.01      0.02       249
    positive       0.65      0.61      0.63      1100

    accuracy                           0.73      3518
   macro avg       0.81      0.50      0.49      3518
weighted avg       0.75      0.73      0.70      3518



**SVM dengan Word2Vector (70/30)**

In [37]:
from sklearn.linear_model import SGDClassifier

svm_4 = SGDClassifier(loss="hinge", random_state=42).fit(X_w2v_train_2, y_train_2)

y_pred_train_svm_4 = svm_4.predict(X_w2v_train_2)
y_pred_test_svm_4 = svm_4.predict(X_w2v_test_2)

accuracy_train_svm_4 = accuracy_score(y_pred_train_svm_4, y_train_2)
accuracy_test_svm_4 = accuracy_score(y_pred_test_svm_4, y_test_2)

print("SVM - accuracy_train_schema_2:", accuracy_train_svm_4)
print("SVM - accuracy_test_schema_2:", accuracy_test_svm_4)

SVM - accuracy_train_schema_2: 0.7688434048083171
SVM - accuracy_test_schema_2: 0.7369717642599962


In [38]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_2, y_pred_test_svm_4, target_names=["negative", "neutral", "positive"]
    )
)

              precision    recall  f1-score   support

    negative       0.77      0.89      0.82      3315
     neutral       1.00      0.01      0.02       388
    positive       0.65      0.60      0.63      1574

    accuracy                           0.74      5277
   macro avg       0.81      0.50      0.49      5277
weighted avg       0.75      0.74      0.71      5277



**Random Forest dengan TF-IDF (80/20)**

In [39]:
from sklearn.ensemble import RandomForestClassifier

rf_1 = RandomForestClassifier().fit(X_tfidf_train_1, y_train_1)

y_pred_train_rf_1 = rf_1.predict(X_tfidf_train_1)
y_pred_test_rf_1 = rf_1.predict(X_tfidf_test_1)

accuracy_train_rf_1 = accuracy_score(y_pred_train_rf_1, y_train_1)
accuracy_test_rf_1 = accuracy_score(y_pred_test_rf_1, y_test_1)

print("Random Forest - accuracy_train_schema_1:", accuracy_train_rf_1)
print("Random Forest - accuracy_test_schema_1:", accuracy_test_rf_1)

Random Forest - accuracy_train_schema_1: 1.0
Random Forest - accuracy_test_schema_1: 0.7765776009096077


In [40]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_1, y_pred_test_rf_1, target_names=["negative", "neutral", "positive"]
    )
)

              precision    recall  f1-score   support

    negative       0.77      0.94      0.85      2169
     neutral       0.62      0.12      0.20       249
    positive       0.80      0.59      0.68      1100

    accuracy                           0.78      3518
   macro avg       0.73      0.55      0.58      3518
weighted avg       0.77      0.78      0.75      3518



**Random Forest dengan TF-IDF (70/30)**

In [41]:
from sklearn.ensemble import RandomForestClassifier

rf_2 = RandomForestClassifier().fit(X_tfidf_train_2, y_train_2)

y_pred_train_rf_2 = rf_2.predict(X_tfidf_train_2)
y_pred_test_rf_2 = rf_2.predict(X_tfidf_test_2)

accuracy_train_rf_2 = accuracy_score(y_pred_train_rf_2, y_train_2)
accuracy_test_rf_2 = accuracy_score(y_pred_test_rf_2, y_test_2)

print("Random Forest - accuracy_train_schema_2:", accuracy_train_rf_2)
print("Random Forest - accuracy_test_schema_2:", accuracy_test_rf_2)

Random Forest - accuracy_train_schema_2: 1.0
Random Forest - accuracy_test_schema_2: 0.7714610574189881


In [42]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_2, y_pred_test_rf_2, target_names=["negative", "neutral", "positive"]
    )
)

              precision    recall  f1-score   support

    negative       0.77      0.95      0.85      3315
     neutral       0.56      0.10      0.17       388
    positive       0.80      0.56      0.66      1574

    accuracy                           0.77      5277
   macro avg       0.71      0.54      0.56      5277
weighted avg       0.76      0.77      0.74      5277



**Random Forest dengan Word2Vector (80/20)**

In [43]:
from sklearn.ensemble import RandomForestClassifier

rf_3 = RandomForestClassifier().fit(X_w2v_train_1, y_train_1)

y_pred_train_rf_3 = rf_3.predict(X_w2v_train_1)
y_pred_test_rf_3 = rf_3.predict(X_w2v_test_1)

accuracy_train_rf_3 = accuracy_score(y_pred_train_rf_3, y_train_1)
accuracy_test_rf_3 = accuracy_score(y_pred_test_rf_3, y_test_1)

print("Random Forest - accuracy_train_schema_2:", accuracy_train_rf_3)
print("Random Forest - accuracy_test_schema_2:", accuracy_test_rf_3)

Random Forest - accuracy_train_schema_2: 1.0
Random Forest - accuracy_test_schema_2: 0.7157475838544628


In [44]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_1, y_pred_test_rf_3, target_names=["negative", "neutral", "positive"]
    )
)

              precision    recall  f1-score   support

    negative       0.74      0.89      0.81      2169
     neutral       0.62      0.03      0.06       249
    positive       0.64      0.52      0.57      1100

    accuracy                           0.72      3518
   macro avg       0.67      0.48      0.48      3518
weighted avg       0.70      0.72      0.68      3518



**Random Forest dengan Word2Vector (70/30)**

In [45]:
from sklearn.ensemble import RandomForestClassifier

rf_4 = RandomForestClassifier().fit(X_w2v_train_2, y_train_2)

y_pred_train_rf_4 = rf_4.predict(X_w2v_train_2)
y_pred_test_rf_4 = rf_4.predict(X_w2v_test_2)

accuracy_train_rf_4 = accuracy_score(y_pred_train_rf_4, y_train_2)
accuracy_test_rf_4 = accuracy_score(y_pred_test_rf_4, y_test_2)

print("Random Forest - accuracy_train_schema_2:", accuracy_train_rf_4)
print("Random Forest - accuracy_test_schema_2:", accuracy_test_rf_4)

Random Forest - accuracy_train_schema_2: 1.0
Random Forest - accuracy_test_schema_2: 0.7138525677468258


In [46]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_2, y_pred_test_rf_4, target_names=["negative", "neutral", "positive"]
    )
)

              precision    recall  f1-score   support

    negative       0.74      0.90      0.81      3315
     neutral       0.57      0.02      0.04       388
    positive       0.64      0.49      0.55      1574

    accuracy                           0.71      5277
   macro avg       0.65      0.47      0.47      5277
weighted avg       0.70      0.71      0.68      5277



In [47]:
results = pd.DataFrame(
    [
        {
            "Model": "SVM",
            "Fitur": "TF-IDF",
            "Schema": "80:20",
            "Accuracy Train": accuracy_train_svm_1 * 100,
            "Accuracy Test": accuracy_test_svm_1 * 100,
        },
        {
            "Model": "SVM",
            "Fitur": "TF-IDF",
            "Schema": "70:30",
            "Accuracy Train": accuracy_train_svm_2 * 100,
            "Accuracy Test": accuracy_test_svm_2 * 100,
        },
        {
            "Model": "SVM",
            "Fitur": "Word2Vec",
            "Schema": "80:20",
            "Accuracy Train": accuracy_train_svm_3 * 100,
            "Accuracy Test": accuracy_test_svm_3 * 100,
        },
        {
            "Model": "SVM",
            "Fitur": "Word2Vec",
            "Schema": "70:30",
            "Accuracy Train": accuracy_train_svm_4 * 100,
            "Accuracy Test": accuracy_test_svm_4 * 100,
        },
        {
            "Model": "Random Forest",
            "Fitur": "TF-IDF",
            "Schema": "80:20",
            "Accuracy Train": accuracy_train_rf_1 * 100,
            "Accuracy Test": accuracy_test_rf_1 * 100,
        },
        {
            "Model": "Random Forest",
            "Fitur": "TF-IDF",
            "Schema": "70:30",
            "Accuracy Train": accuracy_train_rf_2 * 100,
            "Accuracy Test": accuracy_test_rf_2 * 100,
        },
        {
            "Model": "Random Forest",
            "Fitur": "Word2Vec",
            "Schema": "80:20",
            "Accuracy Train": accuracy_train_rf_3 * 100,
            "Accuracy Test": accuracy_test_rf_3 * 100,
        },
        {
            "Model": "Random Forest",
            "Fitur": "Word2Vec",
            "Schema": "70:30",
            "Accuracy Train": accuracy_train_rf_4 * 100,
            "Accuracy Test": accuracy_test_rf_4 * 100,
        },
    ]
)

results

,Model,Fitur,Schema,Accuracy Train,Accuracy Test
0,SVM,TF-IDF,80:20,93.603866,87.123366
1,SVM,TF-IDF,70:30,94.282001,86.450635
2,SVM,Word2Vec,80:20,76.689645,73.166572
3,SVM,Word2Vec,70:30,76.884340,73.697176
4,Random Forest,TF-IDF,80:20,100.000000,77.657760
5,Random Forest,TF-IDF,70:30,100.000000,77.146106
6,Random Forest,Word2Vec,80:20,100.000000,71.574758
7,Random Forest,Word2Vec,70:30,100.000000,71.385257


# 7.Inference

In [69]:
ulasan = [
    "Mobile Legends ini sekarang semakin bagus dan seru dimainkan",
    "Game ini sangat buruk, sering lag dan disconnect",
    "Update terbaru biasa saja, tidak ada perubahan yang signifikan",
    "Grafiknya bagus tetapi sering sekali lag",
    "Saya suka sekali bermain Mobile Legends",
    "Game sampah, matchmaking sangat buruk",
    "Setelah update hanya login dan bermain seperti biasa",
]

**Preprocessing**

In [70]:
def text_preprocessing(doc):
    doc = [text.lower() for text in doc]
    doc = [re.sub(r"@[A-Za-z0-9]+", "", text) for text in doc]
    doc = [re.sub(r"#[A-Za-z0-9]+", "", text) for text in doc]
    doc = [re.sub(r"RT[\s]", "", text) for text in doc]
    doc = [re.sub(r"http\S+", "", text) for text in doc]
    doc = [re.sub(r"[0-9]+", "", text) for text in doc]
    doc = [re.sub(r"[^\w\s]", "", text) for text in doc]
    doc = [
        " ".join([slangwords.get(word.lower(), word) for word in text.split()])
        for text in doc
    ]
    doc = [
        " ".join([word for word in text.split() if word not in stopwords_id])
        for text in doc
    ]
    doc = [word_tokenize(text) for text in doc]
    doc = [[stemmer.stem(token) for token in tokens] for tokens in doc]

    return doc

In [71]:
def inference_sentiment(text):

    # Preprocessing teks
    tokens = text_preprocessing([text])[0]

    # Mengubah token menjadi satu string
    cleaned_text = " ".join(tokens)

    # Feature extraction menggunakan TF-IDF
    vector = tfidfvec.transform([cleaned_text])

    # Prediksi menggunakan SVM terbaik
    prediction = svm_1.predict(vector)[0]

    return prediction

In [72]:
hasil_inference = []

for ulasan in ulasan:

    hasil = inference_sentiment(ulasan)

    hasil_inference.append({"Ulasan": ulasan, "Prediksi Sentimen": hasil})

hasil_inference_df = pd.DataFrame(hasil_inference)

hasil_inference_df

,Ulasan,Prediksi Sentimen
0,Mobile Legends ini sekarang semakin bagus dan ...,negative
1,"Game ini sangat buruk, sering lag dan disconnect",negative
2,"Update terbaru biasa saja, tidak ada perubahan...",negative
3,Grafiknya bagus tetapi sering sekali lag,negative
4,Saya suka sekali bermain Mobile Legends,postive
5,"Game sampah, matchmaking sangat buruk",negative
6,Setelah update hanya login dan bermain seperti...,negative


In [73]:
hasil_inference_df.to_csv("hasil_inference_SVM_TF-IDF_80_20.csv")

# 8.Dokumentasi Hasil
**Membuat CSV dari Hasil Preprocessing dan Latih Model**

In [51]:
results.to_csv("Hasil_Perbandingan_Model_Sentiment_Analysis.csv")

In [52]:
df_clean.to_csv("Ulasan_Apk_Mobile_Legends_Processed.csv")

In [74]:
import joblib
import os

# Membuat folder penyimpanan model
os.makedirs("saved_models", exist_ok=True)
joblib.dump(svm_1, "saved_models/svm_tfidf_80_20.pkl")
joblib.dump(svm_2, "saved_models/svm_tfidf_70_30.pkl")
joblib.dump(svm_3, "saved_models/svm_word2vec_80_20.pkl")
joblib.dump(svm_4, "saved_models/svm_word2vec_70_30.pkl")
joblib.dump(rf_1, "saved_models/rf_tfidf_80_20.pkl")
joblib.dump(rf_2, "saved_models/rf_tfidf_70_30.pkl")
joblib.dump(rf_3, "saved_models/rf_word2vec_80_20.pkl")
joblib.dump(rf_4, "saved_models/rf_word2vec_70_30.pkl")
joblib.dump(tfidfvec, "saved_models/tfidf_vectorizer.pkl")
model.save("saved_models/word2vec.model")

print("Word2Vec berhasil disimpan.")


print("Semua model dan TF-IDF berhasil disimpan.")

Word2Vec berhasil disimpan.
Semua model dan TF-IDF berhasil disimpan.
